In [ ]:
import os
import random
import warnings
import time
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║  1. CONFIGURATION                                          ║
# ╚══════════════════════════════════════════════════════════════╝
class CFG:
    # Auto-resolves to one of:
    #   /kaggle/working/mhist_preprocessed/      (already unpacked)
    #   /kaggle/working/mhist_preprocessed.zip   (will be unzipped)
    #   /kaggle/input/.../mhist_preprocessed/    (kaggle dataset)
    DATA_ROOT       = None   # ← resolved at runtime by resolve_data_root()

    OUT_DIR         = '/kaggle/working/mhist_out'
    BATCH_SIZE      = 32
    FOCAL_GAMMA     = 2.0
    FOCAL_ALPHA     = 0.5     # train is 1:1 balanced at preprocessing
    LABEL_SMOOTH    = 0.05
    SEEDS           = [42, 123, 2024, 7, 1337]
    N_SEEDS         = 5
    WEIGHT_DECAY    = 5e-3
    DROPOUT_2       = 0.3
    SWA_FRAC        = 0.5
    SWA_LR_FACTOR   = 1.0
    TTA_FIVE_CROP   = True
    DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED     = True

# MHIST-specific config: harder task than BACH, so we give it more time.
MHIST_CFG = dict(
    FREEZE_EPOCHS=15,       # +5 vs BACH — head needs more time on subtler task
    UNFREEZE_EPOCHS=60,     # +20 vs BACH — harder task, more headroom
    DROPOUT_1=0.5,
    PATIENCE=20,            # +8 vs BACH — longer plateaus expected on hard task
    LR_P1=3e-4,
    LR_HEAD_P2=1e-4,
    LR_BACKBONE_P2=1e-5,
    WEIGHT_DECAY=5e-3,
    P2_WARMUP=3,
    IMG_SIZE=224,           # native MHIST size — no resize wasted
    AUG_STRENGTH='strong',  # H&E stain variation, same as BACH
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = True
os.makedirs(CFG.OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# Auto-discover the data location
# ─────────────────────────────────────────────────────────────────
def resolve_data_root():
    """Return a directory containing 'train/' and 'test/' (each with
    'benign/' and 'malignant/'), unzipping if necessary."""
    # 1. Already unpacked in /kaggle/working/
    direct = '/kaggle/working/mhist_preprocessed'
    if (os.path.isdir(os.path.join(direct, 'train'))
        and os.path.isdir(os.path.join(direct, 'test'))):
        print(f"  Using already-unpacked folder: {direct}")
        return direct

    # 2. Zip in /kaggle/working/
    zip_path = '/kaggle/working/mhist_preprocessed.zip'
    if os.path.exists(zip_path):
        print(f"  Found zip at {zip_path} — extracting...")
        extract_to = '/kaggle/working/_mhist_extracted'
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        # find the inner mhist_preprocessed folder
        for root, dirs, _ in os.walk(extract_to):
            if 'train' in dirs and 'test' in dirs:
                print(f"  Extracted to: {root}")
                return root
        raise FileNotFoundError(f"Couldn't find train/test inside extracted zip at {extract_to}")

    # 3. Kaggle dataset input
    if os.path.isdir('/kaggle/input'):
        for sub in os.listdir('/kaggle/input'):
            base = os.path.join('/kaggle/input', sub)
            for root, dirs, _ in os.walk(base):
                if 'train' in dirs and 'test' in dirs:
                    # Verify it has benign/malignant
                    if (os.path.isdir(os.path.join(root, 'train', 'benign'))
                        and os.path.isdir(os.path.join(root, 'test', 'malignant'))):
                        print(f"  Using Kaggle input dataset: {root}")
                        return root

    raise FileNotFoundError(
        "Could not find MHIST preprocessed data. Searched:\n"
        "  /kaggle/working/mhist_preprocessed/\n"
        "  /kaggle/working/mhist_preprocessed.zip\n"
        "  /kaggle/input/*/...\n"
        "Run the preprocessor first or upload the zip as a Kaggle dataset.")

print("=" * 60)
print("MHIST RUNNER — RESOLVING DATA LOCATION")
print("=" * 60)
CFG.DATA_ROOT = resolve_data_root()
print(f"  DATA_ROOT = {CFG.DATA_ROOT}\n")

print(f"Device         : {CFG.DEVICE}")
print(f"Dataset        : MHIST (binary, HP vs SSA via official 2175/977 split)")
print(f"DATA_ROOT      : {CFG.DATA_ROOT}")
print(f"Seeds to use   : {CFG.SEEDS[:CFG.N_SEEDS]}  (N={CFG.N_SEEDS})")
print(f"Image size     : {MHIST_CFG['IMG_SIZE']}  (native MHIST resolution)")
print(f"AMP            : {CFG.AMP_ENABLED}")
print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}\n")

# ╔══════════════════════════════════════════════════════════════╗
# ║  2. LOSS                                                   ║
# ╚══════════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    """Focal BCE + label smoothing + alpha class-balancing.
    For MHIST (1:1 balanced at preprocessing), alpha=0.5 = equal weighting.
    """
    smooth_labels = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth_labels, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    alpha_t = torch.where(labels == 1,
                          torch.full_like(labels, CFG.FOCAL_ALPHA),
                          torch.full_like(labels, 1.0 - CFG.FOCAL_ALPHA))
    return (alpha_t * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

# ╔══════════════════════════════════════════════════════════════╗
# ║  3. MODEL (ResNet50 + MS-HRA) — IDENTICAL to v0-10        ║
# ╚══════════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.ca       = ChannelAttention(out_ch)
        self.sa       = SpatialAttention()
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out * self.ca(out)
        out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.mshra  = nn.Sequential(
            MS_HRA_Block(1024, 1024, stride=1),
            MS_HRA_Block(1024, 2048, stride=2))
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(CFG.DROPOUT_2), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        return (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
                list(self.layer2.parameters()) + list(self.layer3.parameters()))
    def new_params(self):
        return list(self.mshra.parameters()) + list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x)
        x = self.layer2(x); x = self.layer3(x)
        x = self.mshra(x);  x = self.gap(x)
        return self.head(x)

class TemperatureScaler(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.raw_T = nn.Parameter(torch.zeros(1))
    @property
    def temperature(self):
        return F.softplus(self.raw_T) + 0.5
    def forward(self, x):
        return self.model(x) / self.temperature
    def calibrate(self, loader, max_outer=15, tol=1e-4):
        self.model.eval()
        opt = torch.optim.LBFGS([self.raw_T], lr=0.05, max_iter=20)
        L, Y = [], []
        with torch.no_grad():
            for imgs, labels, _, _ in loader:
                if CFG.AMP_ENABLED:
                    with torch.cuda.amp.autocast():
                        logits = self.model(imgs.to(CFG.DEVICE)).squeeze(1)
                    L.append(logits.float().cpu())
                else:
                    L.append(self.model(imgs.to(CFG.DEVICE)).squeeze(1).cpu())
                Y.append(labels)
        L = torch.cat(L); Y = torch.cat(Y)
        if roc_auc_score(Y.numpy(), torch.sigmoid(L).numpy()) < 0.5: L = -L
        def nll():
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(
                L / (F.softplus(self.raw_T.cpu()) + 0.5), Y)
            loss.backward(); return loss
        prev_T = float('inf')
        for _ in range(max_outer):
            opt.step(nll)
            cur_T = self.temperature.item()
            if abs(cur_T - prev_T) < tol: break
            prev_T = cur_T
        return self.temperature.item()

# ╔══════════════════════════════════════════════════════════════╗
# ║  4. DATA                                                   ║
# ╚══════════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='strong'):
    """Strong augmentation for MHIST — H&E stain variation between slides."""
    if strength == 'strong':
        cj_b, cj_c, cj_s, cj_h = 0.4, 0.4, 0.3, 0.1
        gray_p   = 0.1
        erase_p  = 0.25
        erase_sc = (0.02, 0.15)
    else:
        cj_b, cj_c, cj_s, cj_h = 0.2, 0.2, 0.15, 0.05
        gray_p   = 0.05
        erase_p  = 0.15
        erase_sc = (0.02, 0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj_b, contrast=cj_c,
                               saturation=cj_s, hue=cj_h),
        transforms.RandomGrayscale(p=gray_p),
        transforms.ToTensor(),
        imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        imagenet_norm])

class MHISTDataset(Dataset):
    """Reads MHIST from <DATA_ROOT>/<split>/{benign,malignant}/."""
    def __init__(self, split, transform=None):
        self.samples   = []
        self.transform = transform
        split_dir = os.path.join(CFG.DATA_ROOT, split)
        if not os.path.isdir(split_dir):
            raise FileNotFoundError(
                f"Split folder not found: {split_dir}. "
                f"Did the preprocessor run successfully?")
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(split_dir, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
        n_b = sum(1 for _,l in self.samples if l == 0)
        n_m = sum(1 for _,l in self.samples if l == 1)
        print(f"  [MHISTDataset] split={split:5s}  dir={split_dir}")
        print(f"  [MHISTDataset]   benign(HP)={n_b}  malignant(SSA)={n_m}  total={len(self.samples)}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img    = Image.open(path).convert('RGB')
        tensor = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return tensor, torch.tensor(lbl, dtype=torch.float32), img, path
    def get_labels(self):
        return [lbl for _, lbl in self.samples]

def _collate(b):
    return (torch.stack([x[0] for x in b]),
            torch.stack([x[1] for x in b]),
            [x[2] for x in b], [x[3] for x in b])

def make_train_loader(ds):
    labels = np.array(ds.get_labels())
    if len(labels) == 0:
        raise RuntimeError("Training dataset is empty! Check preprocessing.")
    class_counts  = np.bincount(labels.astype(int), minlength=2)
    class_counts  = np.maximum(class_counts, 1)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels.astype(int)]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(ds), replacement=True)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True,
                      collate_fn=_collate)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False,
                      collate_fn=_collate)

# ╔══════════════════════════════════════════════════════════════╗
# ║  5. TRAINING + EVAL HELPERS                                ║
# ╚══════════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total = 0.0; n = 0
    for imgs, labels, _, _ in loader:
        imgs   = imgs.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1)
                loss   = focal_loss(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs).squeeze(1)
            loss   = focal_loss(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += loss.item() * imgs.size(0); n += imgs.size(0)
    return total / max(n, 1)

@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    probs, targets = [], []
    for imgs, labels, _, _ in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs.to(CFG.DEVICE, non_blocking=True)).squeeze(1)
        else:
            logits = model(imgs.to(CFG.DEVICE, non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(logits.float()).cpu().numpy())
        targets.extend(labels.numpy())
    return roc_auc_score(targets, probs)

@torch.no_grad()
def tta_predict(m, loader, img_size):
    """11-view TTA: 6 rot/flip + 5 crops."""
    m.eval()
    all_probs, all_labels, all_imgs = [], [], []
    crop_size = int(img_size * 0.9)
    for imgs, labels, orig_imgs, _ in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        views = [imgs,
                 torch.flip(imgs, [3]),
                 torch.flip(imgs, [2]),
                 torch.flip(imgs, [2, 3]),
                 torch.rot90(imgs, 1, [2, 3]),
                 torch.rot90(imgs, 3, [2, 3])]
        if CFG.TTA_FIVE_CROP:
            H, W = imgs.shape[2], imgs.shape[3]
            ch, cw = crop_size, crop_size
            offsets = [((H-ch)//2, (W-cw)//2),
                       (0, 0), (0, W-cw),
                       (H-ch, 0), (H-ch, W-cw)]
            for (top, left) in offsets:
                crop = imgs[:, :, top:top+ch, left:left+cw]
                crop = F.interpolate(crop, size=(H, W), mode='bilinear', align_corners=False)
                views.append(crop)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                preds = sum(torch.sigmoid(m(v).float()) for v in views) / len(views)
        else:
            preds = sum(torch.sigmoid(m(v)) for v in views) / len(views)
        all_probs.extend(preds.squeeze(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_imgs.extend(orig_imgs)
    return np.array(all_probs), np.array(all_labels), all_imgs

def denorm(t):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    return (t.cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()

def accuracy_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    accs = [accuracy_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
    return thresholds[np.argmax(accs)], max(accs)

# ╔══════════════════════════════════════════════════════════════╗
# ║  6. XAI HELPERS                                            ║
# ╚══════════════════════════════════════════════════════════════╝
class GradCAM:
    def __init__(self, model, target_layer, img_size):
        self.model = model
        self.img_size = img_size
        self.gradients = None; self.activations = None
        target_layer.register_forward_hook(
            lambda m,i,o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))
    def __call__(self, img_tensor):
        self.model.eval()
        logit = self.model(img_tensor)
        self.model.zero_grad()
        logit.squeeze().backward()
        weights = self.gradients.mean(dim=[2,3], keepdim=True)
        cam     = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = F.interpolate(cam, (self.img_size, self.img_size),
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def collect_examples(val_ds, y_prob, n=4):
    benign, malignant = [], []
    for i in range(len(val_ds)):
        img_t, lbl, orig, path = val_ds[i]
        prob = float(y_prob[i])
        if int(lbl.item()) == 0 and len(benign) < n:
            benign.append((img_t, orig, prob))
        elif int(lbl.item()) == 1 and len(malignant) < n:
            malignant.append((img_t, orig, prob))
        if len(benign) >= n and len(malignant) >= n: break
    return benign, malignant

def plot_gradcam(model, val_ds, y_prob, img_size, n=4):
    cam_engine = GradCAM(model, model.mshra[1].conv2, img_size)
    benign, malignant = collect_examples(val_ds, y_prob, n)
    if not benign or not malignant:
        print("  [XAI] Not enough samples for Grad-CAM grid; skipping.")
        return
    fig, axes = plt.subplots(4, n, figsize=(4*n, 16))
    if n == 1: axes = axes.reshape(4, 1)
    for col, (img_t, orig, prob) in enumerate(benign):
        orig_np = denorm(img_t)
        with torch.enable_grad():
            hmap = cam_engine(img_t.unsqueeze(0).to(CFG.DEVICE))
        overlay = (0.55*orig_np + 0.45*plt.cm.jet(hmap)[:,:,:3]).clip(0,1)
        axes[0][col].imshow(orig_np)
        axes[0][col].set_title(f"HP (benign)  p={prob:.3f}"); axes[0][col].axis('off')
        axes[1][col].imshow(overlay)
        axes[1][col].set_title("Grad-CAM"); axes[1][col].axis('off')
    for col, (img_t, orig, prob) in enumerate(malignant):
        orig_np = denorm(img_t)
        with torch.enable_grad():
            hmap = cam_engine(img_t.unsqueeze(0).to(CFG.DEVICE))
        overlay = (0.55*orig_np + 0.45*plt.cm.jet(hmap)[:,:,:3]).clip(0,1)
        axes[2][col].imshow(orig_np)
        axes[2][col].set_title(f"SSA (malignant)  p={prob:.3f}"); axes[2][col].axis('off')
        axes[3][col].imshow(overlay)
        axes[3][col].set_title("Grad-CAM"); axes[3][col].axis('off')
    fig.suptitle("Grad-CAM on MHIST  |  p = P(SSA)", fontsize=12)
    plt.tight_layout(); plt.show()

def plot_calibration(y_true, y_prob, n_bins=10):
    edges  = np.linspace(0,1,n_bins+1)
    b_acc, b_conf, b_size = [], [], []
    for i in range(n_bins):
        mask = (y_prob>=edges[i]) & (y_prob<edges[i+1])
        if mask.sum()>0:
            b_acc.append(y_true[mask].mean())
            b_conf.append(y_prob[mask].mean())
            b_size.append(mask.sum())
        else:
            b_acc.append(np.nan); b_conf.append((edges[i]+edges[i+1])/2); b_size.append(0)
    b_acc=np.array(b_acc); b_conf=np.array(b_conf); b_size=np.array(b_size)
    valid = ~np.isnan(b_acc)
    ece   = (b_size[valid]/len(y_true)*np.abs(b_acc[valid]-b_conf[valid])).sum()
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot([0,1],[0,1],'--',color='gray',lw=1.5,label='Perfect')
    axes[0].bar(b_conf[valid], b_acc[valid], width=0.08, alpha=0.75,
                color='steelblue', edgecolor='white', label='Model')
    axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)
    axes[0].set_title(f'Reliability  |  ECE={ece:.4f}')
    axes[0].set_xlabel('Mean predicted probability'); axes[0].set_ylabel('Fraction positive')
    axes[0].legend(fontsize=9)
    axes[1].hist(y_prob[y_true==0],bins=25,alpha=0.65,label='HP (benign)',color='steelblue')
    axes[1].hist(y_prob[y_true==1],bins=25,alpha=0.65,label='SSA (malignant)',color='tomato')
    axes[1].set_title('Confidence distribution'); axes[1].legend()
    plt.suptitle("Calibration  |  MHIST", fontsize=12)
    plt.tight_layout(); plt.show()
    return ece

# ╔══════════════════════════════════════════════════════════════╗
# ║  7. TRAIN ONE SEED                                         ║
# ╚══════════════════════════════════════════════════════════════╝
def train_one_seed(seed, mc, train_loader, val_loader):
    set_seed(seed)
    freeze_epochs   = mc['FREEZE_EPOCHS']
    unfreeze_epochs = mc['UNFREEZE_EPOCHS']
    dropout_1       = mc['DROPOUT_1']
    patience        = mc['PATIENCE']
    lr_p1           = mc['LR_P1']
    lr_head_p2      = mc['LR_HEAD_P2']
    lr_backbone_p2  = mc['LR_BACKBONE_P2']
    weight_decay    = mc['WEIGHT_DECAY']
    p2_warmup       = mc['P2_WARMUP']

    model    = Res_MSHRA(dropout_1=dropout_1).to(CFG.DEVICE)
    scaler   = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    history  = {'loss': [], 'val_auc': [], 'phase': []}
    ckpt     = os.path.join(CFG.OUT_DIR, f'res_mshra_MHIST_s{seed}.pth')

    print(f"\n  [seed {seed}] Phase 1 — frozen backbone ({freeze_epochs} epochs)")
    model.freeze_backbone()
    opt1   = torch.optim.AdamW(model.new_params(), lr=lr_p1, weight_decay=weight_decay)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=freeze_epochs, eta_min=1e-6)
    for epoch in range(1, freeze_epochs + 1):
        loss    = train_one_epoch(model, train_loader, opt1, scaler)
        val_auc = evaluate_auc(model, val_loader)
        sched1.step()
        history['loss'].append(loss); history['val_auc'].append(val_auc); history['phase'].append(1)
        is_best = val_auc > best_auc
        if is_best:
            best_auc = val_auc
            torch.save(model.state_dict(), ckpt)
        print(f"    P1 {epoch:02d}/{freeze_epochs} | Loss:{loss:.4f} | AUC:{val_auc:.4f}"
              + (" ←" if is_best else ""))

    swa_target_start = max(int(unfreeze_epochs * (1 - CFG.SWA_FRAC)), p2_warmup + 1)
    swa_budget       = max(int(unfreeze_epochs * CFG.SWA_FRAC), 5)
    print(f"\n  [seed {seed}] Phase 2 — fine-tune (up to {unfreeze_epochs} ep)")
    print(f"  [seed {seed}]   SWA target_start=ep{swa_target_start}, budget={swa_budget} ep")
    model.unfreeze_backbone()
    patience_ctr = 0
    opt2 = torch.optim.AdamW([
        {'params': model.backbone_params(), 'lr': lr_backbone_p2},
        {'params': model.new_params(),      'lr': lr_head_p2},
    ], weight_decay=weight_decay)
    p2_cosine_epochs = unfreeze_epochs - p2_warmup
    sched2 = torch.optim.lr_scheduler.SequentialLR(opt2, schedulers=[
        torch.optim.lr_scheduler.LinearLR(opt2, start_factor=0.1, end_factor=1.0, total_iters=p2_warmup),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=p2_cosine_epochs, eta_min=1e-7),
    ], milestones=[p2_warmup])

    swa_model = AveragedModel(model)
    swa_lr    = lr_backbone_p2 * CFG.SWA_LR_FACTOR
    swa_sched = SWALR(opt2, swa_lr=swa_lr, anneal_epochs=2, anneal_strategy='linear')
    swa_active = False
    swa_activation_epoch = None

    for epoch in range(1, unfreeze_epochs + 1):
        loss    = train_one_epoch(model, train_loader, opt2, scaler)
        val_auc = evaluate_auc(model, val_loader)

        if not swa_active:
            reason = None
            if epoch >= swa_target_start: reason = "scheduled"
            elif patience_ctr >= patience: reason = "patience_forced"
            if reason is not None:
                swa_active = True
                swa_activation_epoch = epoch
                print(f"    [seed {seed}] SWA activated at P2 ep {epoch} ({reason})")

        if swa_active:
            swa_sched.step()
            swa_model.update_parameters(model)
        else:
            sched2.step()

        history['loss'].append(loss); history['val_auc'].append(val_auc); history['phase'].append(2)

        if not swa_active:
            is_best = val_auc > best_auc
            if is_best:
                best_auc = val_auc; patience_ctr = 0
                torch.save(model.state_dict(), ckpt)
            else:
                patience_ctr += 1
            tag = "←" if is_best else f" (p{patience_ctr}/{patience})"
            swa_tag = ""
        else:
            tag = ""; swa_tag = "[SWA]"

        print(f"    P2 {epoch:02d} {swa_tag} | Loss:{loss:.4f} | AUC:{val_auc:.4f}  {tag}")

        if swa_active and (epoch - swa_activation_epoch + 1) >= swa_budget:
            print(f"    [seed {seed}] SWA budget complete at P2 ep {epoch}.")
            break

    if swa_active:
        print(f"    [seed {seed}] Finalizing SWA: recomputing BN stats...")
        update_bn(train_loader, swa_model, device=CFG.DEVICE)
        swa_auc = evaluate_auc(swa_model, val_loader)
        print(f"    [seed {seed}] SWA AUC={swa_auc:.4f}  vs  best ckpt AUC={best_auc:.4f}")
        if swa_auc > best_auc:
            print(f"    [seed {seed}] SWA wins.")
            chosen = swa_model
        else:
            print(f"    [seed {seed}] Best ckpt wins.")
            model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
            chosen = model
    else:
        model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
        chosen = model

    scaled = TemperatureScaler(chosen).to(CFG.DEVICE)
    T = scaled.calibrate(val_loader)
    print(f"    [seed {seed}] Temperature T={T:.3f}")

    return scaled, history, T

# ╔══════════════════════════════════════════════════════════════╗
# ║  8. TRAIN ALL SEEDS + ENSEMBLE                            ║
# ╚══════════════════════════════════════════════════════════════╝
def train_and_evaluate():
    mc           = MHIST_CFG
    img_size     = mc['IMG_SIZE']
    aug_strength = mc['AUG_STRENGTH']
    n_seeds_mag  = CFG.N_SEEDS

    print(f"\n{'#'*62}")
    print(f"  DATASET: MHIST (HP vs SSA; official 2175/977 train/test split)")
    print(f"  img_size={img_size}  aug={aug_strength}  N_SEEDS={n_seeds_mag}")
    print(f"  freeze={mc['FREEZE_EPOCHS']}  unfreeze={mc['UNFREEZE_EPOCHS']}  "
          f"patience={mc['PATIENCE']}")
    print(f"{'#'*62}")

    t_train = get_train_transform(img_size, strength=aug_strength)
    t_val   = get_val_transform(img_size)
    print("\n  Building datasets...")
    train_ds = MHISTDataset('train', t_train)
    val_ds   = MHISTDataset('test',  t_val)
    train_loader = make_train_loader(train_ds)
    val_loader   = make_val_loader(val_ds)
    print()

    seed_probs_list = []
    seed_results    = []
    first_model     = None
    first_history   = None
    y_true_ref      = None
    seeds_to_run    = CFG.SEEDS[:n_seeds_mag]

    for seed in seeds_to_run:
        idx = seeds_to_run.index(seed) + 1
        print(f"\n  ━━━ MHIST  seed={seed}  ({idx}/{len(seeds_to_run)}) ━━━")
        t0 = time.time()
        scaled, history, T = train_one_seed(seed, mc, train_loader, val_loader)
        y_prob, y_true, _ = tta_predict(scaled, val_loader, img_size)
        seed_auc = roc_auc_score(y_true, y_prob)
        print(f"  [seed {seed}] standalone TTA AUC = {seed_auc:.4f}   "
              f"({(time.time()-t0)/60:.1f} min)")
        seed_probs_list.append(y_prob)
        seed_results.append({'seed': seed, 'auc': seed_auc, 'T': T})
        if first_model is None:
            first_model   = scaled.model
            first_history = history
            y_true_ref    = y_true

    y_prob = np.mean(np.stack(seed_probs_list, axis=0), axis=0)
    y_true = y_true_ref
    roc_auc_val = roc_auc_score(y_true, y_prob)
    auprc_val   = average_precision_score(y_true, y_prob)
    seed_auc_str = ", ".join("{:.4f}".format(r['auc']) for r in seed_results)
    print(f"\n  ═══ MHIST ENSEMBLE AUC = {roc_auc_val:.4f}  "
          f"AUPRC = {auprc_val:.4f}  (seeds: [{seed_auc_str}]) ═══")

    best_thr, best_acc = accuracy_optimal_threshold(y_true, y_prob)
    y_pred = (y_prob >= best_thr).astype(int)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0.0
    bal_acc = (sens + spec) / 2
    ece = plot_calibration(y_true, y_prob)

    print(f"\n{'='*62}")
    print(f"  RESULTS: MHIST  (ensemble of {n_seeds_mag} seeds)")
    print(f"{'='*62}")
    print(f"  ROC-AUC       : {roc_auc_val:.4f}")
    print(f"  AUPRC         : {auprc_val:.4f}")
    print(f"  Accuracy      : {best_acc*100:.2f}%  (thr={best_thr:.4f})")
    print(f"  Bal Accuracy  : {bal_acc*100:.2f}%")
    print(f"  Sensitivity   : {sens:.4f}")
    print(f"  Specificity   : {spec:.4f}")
    print(f"  Precision     : {precision_score(y_true,y_pred):.4f}")
    print(f"  F1-score      : {f1_score(y_true,y_pred):.4f}")
    print(f"  ECE           : {ece:.4f}")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"  Per-seed AUCs : [{seed_auc_str}]")
    print(f"  Per-seed std  : {np.std([r['auc'] for r in seed_results]):.4f}")
    print(classification_report(y_true, y_pred, target_names=['HP (Benign)','SSA (Malignant)']))

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    sns.heatmap(confusion_matrix(y_true,y_pred), annot=True, fmt='d', cmap='Purples',
                ax=axes[0], xticklabels=['HP','SSA'],
                yticklabels=['HP','SSA'])
    axes[0].set_title('MHIST confusion matrix')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={roc_auc_val:.4f}')
    axes[1].plot([0,1],[0,1],'k--',lw=1); axes[1].legend(loc='lower right')
    axes[1].set_title('MHIST ROC curve (ensemble)')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[2].hist(y_prob[y_true==0],bins=25,alpha=0.65,label='HP',color='steelblue')
    axes[2].hist(y_prob[y_true==1],bins=25,alpha=0.65,label='SSA',color='tomato')
    axes[2].axvline(best_thr,color='black',linestyle='--',label=f'thr={best_thr:.2f}')
    axes[2].set_title('MHIST probability distribution'); axes[2].legend()
    plt.suptitle(f"Res-MSHRA on MHIST  |  AUC={roc_auc_val:.4f}  "
                 f"BalAcc={bal_acc*100:.1f}%  ({n_seeds_mag}-seed ensemble)", fontsize=13)
    plt.tight_layout(); plt.show()

    if first_history:
        phases = np.array(first_history['phase'])
        ep = np.arange(1, len(first_history['loss'])+1)
        p1_end = int(np.where(phases==1)[0][-1]) + 1 if (phases==1).any() else 0
        fig, axes = plt.subplots(1,2,figsize=(13,4))
        axes[0].plot(ep, first_history['loss'], color='steelblue')
        if p1_end: axes[0].axvline(p1_end,color='gray',linestyle=':',alpha=0.7,label='P1→P2')
        axes[0].set_title(f'MHIST training loss (seed {seeds_to_run[0]})'); axes[0].legend()
        axes[1].plot(ep, first_history['val_auc'], color='darkorange')
        if p1_end: axes[1].axvline(p1_end,color='gray',linestyle=':',alpha=0.7,label='P1→P2')
        axes[1].axhline(roc_auc_val,linestyle='--',color='red',alpha=0.5,
                        label=f'Ensemble AUC={roc_auc_val:.4f}')
        axes[1].set_title(f'MHIST val AUC (seed {seeds_to_run[0]})'); axes[1].legend()
        plt.tight_layout(); plt.show()

    xai_model = first_model.module if isinstance(first_model, AveragedModel) else first_model
    print(f"\n  Running Grad-CAM XAI for MHIST (using seed-{seeds_to_run[0]} model)...")
    plot_gradcam(xai_model, val_ds, y_prob, img_size)

    return {
        'dataset': 'MHIST', 'auc': roc_auc_val, 'auprc': auprc_val,
        'acc': best_acc, 'bal_acc': bal_acc,
        'sensitivity': sens, 'specificity': spec,
        'precision': precision_score(y_true,y_pred),
        'f1': f1_score(y_true,y_pred), 'ece': ece,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'threshold': best_thr, 'fpr': fpr, 'tpr': tpr,
        'n_train': len(train_ds), 'n_test': len(val_ds),
        'per_seed_aucs': [r['auc'] for r in seed_results],
        'per_seed_std' : float(np.std([r['auc'] for r in seed_results])),
    }

# ╔══════════════════════════════════════════════════════════════╗
# ║  9. RUN                                                    ║
# ╚══════════════════════════════════════════════════════════════╝
overall_t0 = time.time()
result = train_and_evaluate()
print(f"\n⏱  Total wall time: {(time.time()-overall_t0)/60:.1f} min")

print(f"\n{'='*100}")
print("  PAPER-READY ROW — MHIST")
print(f"{'='*100}")
r = result
print(f"  Dataset : MHIST (HP vs SSA, official 2175/977 split)")
print(f"  Train/Test = {r['n_train']}/{r['n_test']}   N_seeds = {len(r['per_seed_aucs'])}")
print(f"  AUC = {r['auc']:.4f} ± {r['per_seed_std']:.4f}   AUPRC = {r['auprc']:.4f}")
print(f"  BalAcc = {r['bal_acc']*100:.2f}%   Sens = {r['sensitivity']:.4f}   "
      f"Spec = {r['specificity']:.4f}")
print(f"  F1 = {r['f1']:.4f}   Precision = {r['precision']:.4f}   ECE = {r['ece']:.4f}")
print(f"{'='*100}")
print(f"  Checkpoints saved in {CFG.OUT_DIR}/")
